# 02 — Transformação Silver das dimensões

## Objetivo

Este notebook consolida e padroniza as entidades dimensionais do case:

- regiões
- canais
- clientes
- vendedores
- produtos

## Responsabilidades

- normalizar identificadores e domínios
- converter datas e valores para tipos adequados
- resolver duplicidades por regras específicas
- preservar os valores recebidos da origem
- criar indicadores de qualidade
- manter rastreabilidade até os registros da camada Bronze
- disponibilizar uma versão consolidada por chave de negócio

As regras aplicadas neste notebook foram definidas a partir dos problemas identificados no notebook 00_exploracao_fontes.

In [0]:
# Precisa instalar essa dependencia não é nativa do Databricks
%pip install openpyxl

In [0]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set(
    "spark.sql.session.timeZone",
    "America/Sao_Paulo"
)

SCHEMA_BRONZE = "case_data_engineer_bronze"
SCHEMA_SILVER = "case_data_engineer_silver"

SILVER_TIMESTAMP = datetime.now(timezone.utc)

print(f"Timestamp Silver UTC: {SILVER_TIMESTAMP}")

In [0]:
# ============================================================
# LOCALIZAÇÃO DO CATÁLOGO
# ============================================================

catalogos_disponiveis = [
    linha["catalog"]
    for linha in spark.sql("SHOW CATALOGS").collect()
]

catalogos_ignorados = {
    "system",
    "samples",
    "hive_metastore"
}

CATALOG = None

for catalogo in catalogos_disponiveis:

    if catalogo.lower() in catalogos_ignorados:
        continue

    try:
        (
            spark.table(
                f"`{catalogo}`."
                f"`{SCHEMA_BRONZE}`."
                f"`bronze_clientes`"
            )
            .limit(1)
            .collect()
        )

        CATALOG = catalogo
        break

    except Exception:
        continue

if CATALOG is None:
    raise RuntimeError(
        "A camada Bronze não foi localizada, execute primeiro o notebook 01_ingestao_bronze!"
    )

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    `{CATALOG}`.`{SCHEMA_SILVER}`
    COMMENT 'Camada Silver do case técnico de Engenharia de Dados'
    """
)

print(f"Catálogo localizado: {CATALOG}")
print(f"Schema Bronze: {CATALOG}.{SCHEMA_BRONZE}")
print(f"Schema Silver: {CATALOG}.{SCHEMA_SILVER}")

In [0]:
# ============================================================
# FUNÇÕES DE NORMALIZAÇÃO
# ============================================================

CARACTERES_ACENTUADOS = (
    "ÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ"
    "áàãâäéèêëíìîïóòõôöúùûüçÑñ"
)

CARACTERES_SEM_ACENTO = (
    "AAAAAEEEEIIIIOOOOOUUUUC"
    "aaaaaeeeeiiiiooooouuuucNn"
)

# Remove os principais acentos encontrados nas fontes
def remover_acentos(coluna):
    return F.translate(
        coluna,
        CARACTERES_ACENTUADOS,
        CARACTERES_SEM_ACENTO
    )

# Remove espaços laterais e converte identificadores para caixa alta
def normalizar_id(coluna):
    return F.upper(
        F.trim(
            coluna.cast("string")
        )
    )

# Padroniza valores categóricos para caixa alta e underscore
def normalizar_dominio(coluna):
    valor = remover_acentos(
        F.upper(
            F.trim(
                coluna.cast("string")
            )
        )
    )

    valor = F.regexp_replace(
        valor,
        r"[^A-Z0-9]+",
        "_"
    )

    return F.regexp_replace(
        valor,
        r"(^_+|_+$)",
        ""
    )

# Converte representações comuns de verdadeiro e falso
def normalizar_booleano(coluna):
    valor = normalizar_dominio(coluna)

    return (
        F.when(
            valor.isin(
                "1",
                "SIM",
                "S",
                "TRUE",
                "ATIVO"
            ),
            F.lit(True)
        )
        .when(
            valor.isin(
                "0",
                "NAO",
                "N",
                "FALSE",
                "INATIVO"
            ),
            F.lit(False)
        )
        .otherwise(
            F.lit(None).cast("boolean")
        )
    )

# Converte siglas, nomes completos e abreviações de estados
def normalizar_uf(coluna):
    valor = normalizar_dominio(coluna)

    return (
        F.when(
            valor.isin("SP", "SAO_PAULO"),
            "SP"
        )
        .when(
            valor.isin("RJ", "RIO_DE_JANEIRO"),
            "RJ"
        )
        .when(
            valor.isin("MG", "MINAS_GERAIS"),
            "MG"
        )
        .when(
            valor.isin("PR", "PARANA"),
            "PR"
        )
        .when(
            valor.isin(
                "SC",
                "SANTA_CATARINA",
                "STA_CATARINA",
                "S_CATARINA"
            ),
            "SC"
        )
        .when(valor == "AM", "AM")
        .when(valor == "BA", "BA")
        .when(valor == "GO", "GO")
        .otherwise(
            F.lit(None).cast("string")
        )
    )

# Consolida variações dos códigos regionais
def normalizar_codigo_regiao(coluna):
    valor = normalizar_dominio(coluna)

    return (
        F.when(valor == "SUL", "S")
        .when(valor == "SUDESTE", "SE")
        .when(valor == "NORDESTE", "NE")
        .when(valor == "NORTE", "N")
        .when(valor == "CENTRO_OESTE", "CO")
        .otherwise(valor)
    )


print("Funções de normalização OK!")

In [0]:
# ============================================================
# LEITURA DAS TABELAS BRONZE
# ============================================================

bronze_regioes = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_regioes`"
)

bronze_canais = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_canais`"
)

print(f"Regiões Bronze: {bronze_regioes.count()} registros")
print(f"Canais Bronze:  {bronze_canais.count()} registros")

In [0]:
# ============================================================
# TRANSFORMAÇÃO SILVER — REGIÕES
# ============================================================

janela_regiao = Window.partitionBy(
    "regional_code"
)

janela_prioridade_regiao = (
    Window
    .partitionBy("regional_code")
    .orderBy(
        F.desc("_quality_score"),
        F.desc("_ingestion_timestamp_utc"),
        F.asc("_source_record_hash")
    )
)

silver_regioes_preparacao = (
    bronze_regioes
    .select(
        F.col("regional_code").alias(
            "regional_code_original"
        ),
        F.col("regional_name").alias(
            "regional_name_original"
        ),
        F.col("state").alias(
            "state_original"
        ),
        F.col("manager_name").alias(
            "manager_name_original"
        ),
        F.col("active_flag").alias(
            "active_flag_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "regional_code",
        normalizar_codigo_regiao(
            F.col("regional_code_original")
        )
    )
    .withColumn(
        "regional_name",
        F.when(
            F.col("regional_code") == "XX",
            F.lit("Não identificada")
        )
        .otherwise(
            F.trim(
                F.col("regional_name_original")
            )
        )
    )
    .withColumn(
        "state_code",
        normalizar_uf(
            F.col("state_original")
        )
    )
    .withColumn(
        "manager_name",
        F.when(
            F.trim(
                F.col("manager_name_original")
            ) == "",
            F.lit(None)
        )
        .otherwise(
            F.trim(
                F.col("manager_name_original")
            )
        )
    )
    .withColumn(
        "is_active",
        normalizar_booleano(
            F.col("active_flag_original")
        )
    )
    .withColumn(
        "dq_unknown_region",
        F.col("regional_code") == "XX"
    )
    .withColumn(
        "dq_missing_state",
        F.col("state_code").isNull()
        & (F.col("regional_code") != "XX")
    )
    .withColumn(
        "dq_duplicate_key",
        F.count("*").over(janela_regiao) > 1
    )
    .withColumn(
        "_quality_score",
        F.when(
            F.col("is_active") == True,
            20
        ).otherwise(0)
        +
        F.when(
            F.col("state_code").isNotNull(),
            10
        ).otherwise(0)
        +
        F.when(
            normalizar_id(
                F.col("state_original")
            ) == F.col("state_code"),
            3
        ).otherwise(0)
        +
        F.when(
            normalizar_id(
                F.col("regional_code_original")
            ) == F.col("regional_code"),
            2
        ).otherwise(0)
        +
        F.when(
            F.col("manager_name").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_regiao
        )
    )
)

silver_regioes = (
    silver_regioes_preparacao
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_TIMESTAMP)
    )
    .drop(
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_regioes
    .orderBy("regional_code")
)

In [0]:
# ============================================================
# VALIDAÇÃO — REGIÕES
# ============================================================

quantidade_regioes_bronze = bronze_regioes.count()
quantidade_regioes_silver = silver_regioes.count()

duplicidades_regioes_silver = (
    silver_regioes
    .groupBy("regional_code")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

chaves_nulas_regioes = (
    silver_regioes
    .filter(
        F.col("regional_code").isNull()
        | (F.col("regional_code") == "")
    )
    .count()
)

print(f"Bronze: {quantidade_regioes_bronze}")
print(f"Silver: {quantidade_regioes_silver}")
print(f"Duplicidades na Silver: {duplicidades_regioes_silver}")
print(f"Chaves nulas na Silver: {chaves_nulas_regioes}")

if duplicidades_regioes_silver > 0:
    raise RuntimeError(
        "A Silver de regiões ainda possui chaves duplicadas."
    )

if chaves_nulas_regioes > 0:
    raise RuntimeError(
        "A Silver de regiões possui chaves de negócio nulas."
    )

In [0]:
# ============================================================
# TRANSFORMAÇÃO SILVER — CANAIS
# ============================================================

# Seleção e normalização inicial
silver_canais_base = (
    bronze_canais
    .select(
        F.col("id_canal").alias(
            "channel_id_original"
        ),
        F.col("nome_canal").alias(
            "channel_name_original"
        ),
        F.col("tipo_canal").alias(
            "channel_type_original"
        ),
        F.col("ativo").alias(
            "active_original"
        ),
        F.col("observacao").alias(
            "observation_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "channel_id",
        normalizar_id(
            F.col("channel_id_original")
        )
    )
    .withColumn(
        "_channel_name_code",
        normalizar_dominio(
            F.col("channel_name_original")
        )
    )
    .withColumn(
        "_channel_name_code",
        F.when(
            F.col("_channel_name_code") == "ECOMMERCE",
            F.lit("E_COMMERCE")
        )
        .otherwise(
            F.col("_channel_name_code")
        )
    )
    .withColumn(
        "channel_name",
        F.when(
            F.col("_channel_name_code") == "INSIDE_SALES",
            F.lit("Inside Sales")
        )
        .when(
            F.col("_channel_name_code") == "FIELD_SALES",
            F.lit("Field Sales")
        )
        .when(
            F.col("_channel_name_code") == "MARKETPLACE",
            F.lit("Marketplace")
        )
        .when(
            F.col("_channel_name_code") == "PARCEIROS",
            F.lit("Parceiros")
        )
        .when(
            F.col("_channel_name_code") == "REVENDAS",
            F.lit("Revendas")
        )
        .when(
            F.col("_channel_name_code") == "E_COMMERCE",
            F.lit("E-commerce")
        )
        .otherwise(
            F.lit("Não informado")
        )
    )
    .withColumn(
        "channel_type",
        normalizar_dominio(
            F.col("channel_type_original")
        )
    )
    .withColumn(
        "is_active",
        normalizar_booleano(
            F.col("active_original")
        )
    )
)

# Métricas por chave
metricas_canais = (
    silver_canais_base
    .groupBy("channel_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("_channel_name_code"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_channel_names"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("channel_type"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_channel_types"
        )
    )
)

# Junção das métricas e criação dos indicadores
silver_canais_preparacao = (
    silver_canais_base
    .join(
        metricas_canais,
        on="channel_id",
        how="left"
    )
    .withColumn(
        "dq_missing_name",
        F.col("channel_name_original").isNull()
        | (
            F.trim(
                F.col("channel_name_original")
            ) == ""
        )
    )
    .withColumn(
        "dq_missing_active_flag",
        F.col("is_active").isNull()
    )
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_channel_names") > 1
        )
        |
        (
            F.col("_distinct_channel_types") > 1
        )
    )
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_name"),
            20
        ).otherwise(0)
        +
        F.when(
            F.col("channel_type").isNotNull(),
            10
        ).otherwise(0)
        +
        F.when(
            F.col("is_active").isNotNull(),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.upper(
                F.coalesce(
                    F.col("observation_original"),
                    F.lit("")
                )
            ).contains("DUPLICADO"),
            30
        ).otherwise(0)
    )
)

# Priorização e deduplicação
janela_prioridade_canal = (
    Window
    .partitionBy("channel_id")
    .orderBy(
        F.desc("_quality_score"),
        F.desc("_ingestion_timestamp_utc"),
        F.asc("_source_record_hash")
    )
)

silver_canais = (
    silver_canais_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_canal
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_TIMESTAMP)
    )
    .drop(
        "_channel_name_code",
        "_records_by_key",
        "_distinct_channel_names",
        "_distinct_channel_types",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_canais.orderBy("channel_id")
)

In [0]:
# ============================================================
# VALIDAÇÃO — CANAIS
# ============================================================

quantidade_canais_bronze = bronze_canais.count()
quantidade_canais_silver = silver_canais.count()

duplicidades_canais_silver = (
    silver_canais
    .groupBy("channel_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

chaves_nulas_canais = (
    silver_canais
    .filter(
        F.col("channel_id").isNull()
        | (F.col("channel_id") == "")
    )
    .count()
)

print(f"Bronze: {quantidade_canais_bronze}")
print(f"Silver: {quantidade_canais_silver}")
print(f"Duplicidades na Silver: {duplicidades_canais_silver}")
print(f"Chaves nulas na Silver: {chaves_nulas_canais}")

if duplicidades_canais_silver > 0:
    raise RuntimeError(
        "A Silver de canais ainda possui chaves duplicadas."
    )

if chaves_nulas_canais > 0:
    raise RuntimeError(
        "A Silver de canais possui chaves de negócio nulas."
    )

In [0]:
# ============================================================
# DIMENSÕES REGIOES E CANAIS
# ============================================================

TABELA_SILVER_REGIOES = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_regioes`"
)

TABELA_SILVER_CANAIS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_canais`"
)

(
    silver_regioes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_REGIOES)
)

(
    silver_canais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_CANAIS)
)

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_regioes"
)

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_canais"
)

In [0]:
# ============================================================
# AUDITORIA PARCIAL DAS DIMENSÕES
# ============================================================

resultado_auditoria_dimensoes = [
    (
        "silver_regioes",
        quantidade_regioes_bronze,
        quantidade_regioes_silver,
        quantidade_regioes_bronze
        - quantidade_regioes_silver,
        duplicidades_regioes_silver,
        chaves_nulas_regioes,
        "SUCESSO"
    ),
    (
        "silver_canais",
        quantidade_canais_bronze,
        quantidade_canais_silver,
        quantidade_canais_bronze
        - quantidade_canais_silver,
        duplicidades_canais_silver,
        chaves_nulas_canais,
        "SUCESSO"
    )
]

schema_auditoria_dimensoes = """
    tabela STRING,
    quantidade_bronze LONG,
    quantidade_silver LONG,
    registros_consolidados LONG,
    duplicidades_saida LONG,
    chaves_nulas_saida LONG,
    status STRING
"""

df_auditoria_dimensoes = spark.createDataFrame(
    resultado_auditoria_dimensoes,
    schema=schema_auditoria_dimensoes
)

display(df_auditoria_dimensoes)

In [0]:
# ============================================================
# FUNÇÕES AUXILIARES — CLIENTES
# ============================================================

FORMATOS_TIMESTAMP = [
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss.SSS",
    "yyyy/MM/dd HH:mm:ss",
    "dd/MM/yyyy HH:mm:ss",
    "dd/MM/yyyy HH:mm",
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy"
]

# Tenta converter uma coluna utilizando os formatos identificados
def converter_timestamp_flexivel_coluna(coluna):
    valor = F.trim(
        coluna.cast("string")
    )

    return F.coalesce(
        *[
            F.try_to_timestamp(
                valor,
                F.lit(formato)
            )
            for formato in FORMATOS_TIMESTAMP
        ]
    )

# Remove espaços laterais e converte o e-mail para letras minúsculas
def normalizar_email(coluna):
    return F.lower(
        F.trim(
            coluna.cast("string")
        )
    )

# Retorna a UF esperada para as cidades presentes no case
def uf_esperada_por_cidade(coluna):
    cidade = normalizar_dominio(coluna)
    return (
        F.when(
            cidade.isin(
                "SAO_PAULO",
                "CAMPINAS",
                "RIBEIRAO_PRETO"
            ),
            F.lit("SP")
        )
        .when(
            cidade.isin(
                "RIO_DE_JANEIRO",
                "NITEROI",
                "VOLTA_REDONDA"
            ),
            F.lit("RJ")
        )
        .when(
            cidade.isin(
                "BELO_HORIZONTE",
                "CONTAGEM",
                "UBERLANDIA"
            ),
            F.lit("MG")
        )
        .when(
            cidade.isin(
                "CURITIBA",
                "LONDRINA",
                "MARINGA"
            ),
            F.lit("PR")
        )
        .when(
            cidade.isin(
                "FLORIANOPOLIS",
                "JOINVILLE",
                "BLUMENAU"
            ),
            F.lit("SC")
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )


print("Funções auxiliares de clientes disponíveis")

In [0]:
# ============================================================
# LEITURA DA BRONZE — CLIENTES
# ============================================================

bronze_clientes = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_clientes`"
)

quantidade_clientes_bronze = (
    bronze_clientes.count()
)

print(
    f"Clientes Bronze: "
    f"{quantidade_clientes_bronze} registros"
)

bronze_clientes.printSchema()

In [0]:
# ============================================================
# PREPARAÇÃO SILVER — CLIENTES
# ============================================================

REGEX_EMAIL_VALIDO = (
    r"^[A-Za-z0-9._%+\-]+"
    r"@[A-Za-z0-9.\-]+"
    r"\.[A-Za-z]{2,}$"
)


silver_clientes_base = (
    bronze_clientes
    .select(
        F.col("customer_id").alias(
            "customer_id_original"
        ),
        F.col("nome_cliente").alias(
            "customer_name_original"
        ),
        F.col("segmento").alias(
            "segment_original"
        ),
        F.col("porte").alias(
            "customer_size_original"
        ),
        F.col("cidade").alias(
            "city_original"
        ),
        F.col("estado").alias(
            "state_original"
        ),
        F.col("data_cadastro").alias(
            "registration_date_original"
        ),
        F.col("email").alias(
            "email_original"
        ),
        F.col("status_cliente").alias(
            "customer_status_original"
        ),
        F.col("updated_at").alias(
            "updated_at_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "customer_id",
        normalizar_id(
            F.col("customer_id_original")
        )
    )
    .withColumn(
        "customer_name",
        F.trim(
            F.col("customer_name_original")
        )
    )
    .withColumn(
        "segment",
        normalizar_dominio(
            F.col("segment_original")
        )
    )
    .withColumn(
        "customer_size",
        normalizar_dominio(
            F.col("customer_size_original")
        )
    )
    .withColumn(
        "city_name",
        F.trim(
            F.col("city_original")
        )
    )
    .withColumn(
        "state_code",
        normalizar_uf(
            F.col("state_original")
        )
    )
    .withColumn(
        "_expected_state_code",
        uf_esperada_por_cidade(
            F.col("city_original")
        )
    )
    .withColumn(
        "registration_date",
        converter_timestamp_flexivel_coluna(
            F.col("registration_date_original")
        ).cast("date")
    )
    .withColumn(
        "email",
        normalizar_email(
            F.col("email_original")
        )
    )
    .withColumn(
        "customer_status",
        normalizar_dominio(
            F.col("customer_status_original")
        )
    )
    .withColumn(
        "updated_at",
        converter_timestamp_flexivel_coluna(
            F.col("updated_at_original")
        )
    )
)

In [0]:
# ============================================================
# CLIENTES- Calculo de  conflitos por chave
# ============================================================

metricas_clientes = (
    silver_clientes_base
    .groupBy("customer_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("customer_name"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_names"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("segment"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_segments"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("state_code"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_states"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("email"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_emails"
        )
    )
)

In [0]:
# ============================================================
# CONSOLIDAÇÃO SILVER — CLIENTES
# ============================================================

silver_clientes_preparacao = (
    silver_clientes_base
    .join(
        metricas_clientes,
        on="customer_id",
        how="left"
    )
    .withColumn(
        "dq_missing_name",
        F.col("customer_name").isNull()
        | (F.trim(F.col("customer_name")) == "")
    )
    .withColumn(
        "dq_missing_segment",
        F.col("segment").isNull()
        | (F.col("segment") == "")
    )
    .withColumn(
        "dq_missing_customer_size",
        F.col("customer_size").isNull()
        | (F.col("customer_size") == "")
    )
    .withColumn(
        "dq_missing_state",
        F.col("state_code").isNull()
    )
    .withColumn(
        "dq_state_city_mismatch",
        F.col("_expected_state_code").isNotNull()
        & F.col("state_code").isNotNull()
        & (
            F.col("_expected_state_code")
            != F.col("state_code")
        )
    )
    .withColumn(
        "dq_missing_email",
        F.col("email").isNull()
        | (F.col("email") == "")
    )
    .withColumn(
        "dq_invalid_email",
        F.col("email").isNotNull()
        & (F.col("email") != "")
        & (~F.col("email").rlike(REGEX_EMAIL_VALIDO))
    )
    .withColumn(
        "dq_invalid_registration_date",
        F.col("registration_date_original").isNotNull()
        & (
            F.trim(
                F.col("registration_date_original")
            ) != ""
        )
        & F.col("registration_date").isNull()
    )
    .withColumn(
        "dq_invalid_updated_at",
        F.col("updated_at_original").isNotNull()
        & (
            F.trim(
                F.col("updated_at_original")
            ) != ""
        )
        & F.col("updated_at").isNull()
    )
    .withColumn(
        "dq_missing_status",
        F.col("customer_status").isNull()
        | (F.col("customer_status") == "")
    )
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_names") > 1
        )
        |
        (
            F.col("_distinct_segments") > 1
        )
        |
        (
            F.col("_distinct_states") > 1
        )
        |
        (
            F.col("_distinct_emails") > 1
        )
    )
)

In [0]:
# ============================================================
# CLIENTES - PONTUAÇÃO DE QUALIDADE
# ============================================================

silver_clientes_preparacao = (
    silver_clientes_preparacao
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_name"),
            20
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_segment"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_customer_size"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_state"),
            10
        ).otherwise(0)
        +
        F.when(
            (
                F.col("_expected_state_code")
                == F.col("state_code")
            ),
            25
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_email")
            & ~F.col("dq_invalid_email"),
            20
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_invalid_registration_date"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_invalid_updated_at"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_status"),
            5
        ).otherwise(0)
    )

)

In [0]:
# ============================================================
# CLIENTES - DEDUPLICAÇÃO
# ============================================================

janela_prioridade_cliente = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.desc("_quality_score"),
        F.desc_nulls_last("updated_at"),
        F.desc("_ingestion_timestamp_utc"),
        F.asc("_source_record_hash")
    )
)


silver_clientes = (
    silver_clientes_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_cliente
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_TIMESTAMP)
    )
    .drop(
        "_expected_state_code",
        "_records_by_key",
        "_distinct_names",
        "_distinct_segments",
        "_distinct_states",
        "_distinct_emails",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_clientes.orderBy("customer_id")
)

In [0]:
# ============================================================
# CLIENTES - VALIDAÇÃO
# ============================================================

quantidade_clientes_silver = (
    silver_clientes.count()
)

quantidade_chaves_distintas_esperada = (
    bronze_clientes
    .select(
        normalizar_id(
            F.col("customer_id")
        ).alias("customer_id")
    )
    .distinct()
    .count()
)

duplicidades_clientes_silver = (
    silver_clientes
    .groupBy("customer_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_clientes = (
    silver_clientes
    .filter(
        F.col("customer_id").isNull()
        | (F.col("customer_id") == "")
    )
    .count()
)

print(
    f"Bronze: "
    f"{quantidade_clientes_bronze}"
)

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_chaves_distintas_esperada}"
)

print(
    f"Silver: "
    f"{quantidade_clientes_silver}"
)

print(
    f"Registros consolidados: "
    f"{quantidade_clientes_bronze - quantidade_clientes_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_clientes_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_clientes}"
)


if (
    quantidade_clientes_silver
    != quantidade_chaves_distintas_esperada
):
    raise RuntimeError(
        "A quantidade da Silver não corresponde as chaves distintas da Bronze"
    )

if duplicidades_clientes_silver > 0:
    raise RuntimeError(
        "A Silver de clientes ainda possui duplicidades"
    )

if chaves_nulas_clientes > 0:
    raise RuntimeError(
        "A Silver de clientes possui chaves nulas"
    )

In [0]:
# ============================================================
# CONFERÊNCIA DOS CLIENTES DUPLICADOS
# ============================================================

display(
    silver_clientes
    .filter(
        F.col("dq_duplicate_key") == True
    )
    .select(
        "customer_id",
        "customer_name",
        "segment",
        "city_name",
        "state_code",
        "email",
        "customer_status",
        "updated_at",
        "dq_state_city_mismatch",
        "dq_invalid_email",
        "dq_conflicting_duplicate"
    )
    .orderBy("customer_id")
)

In [0]:
# ============================================================
# SILVER — CLIENTES
# ============================================================

TABELA_SILVER_CLIENTES = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_clientes`"
)

(
    silver_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_CLIENTES)
)

quantidade_clientes_persistidos = (
    spark.table(TABELA_SILVER_CLIENTES)
    .count()
)

if (
    quantidade_clientes_persistidos
    != quantidade_clientes_silver
):
    raise RuntimeError(
        "A quantidade persistida de clientes diverge da quantidade do DataFrame Silver"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_clientes"
)

print(
    f"Registros persistidos: "
    f"{quantidade_clientes_persistidos}"
)

In [0]:
# ============================================================
# LEITURA DA BRONZE — VENDEDORES
# ============================================================

bronze_vendedores = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_vendedores`"
)

quantidade_vendedores_bronze = (
    bronze_vendedores.count()
)

print(
    f"Vendedores Bronze: "
    f"{quantidade_vendedores_bronze} registros"
)

bronze_vendedores.printSchema()

In [0]:
# ============================================================
# PREPARAÇÃO SILVER — VENDEDORES
# ============================================================

silver_vendedores_base = (
    bronze_vendedores
    .select(
        F.col("seller_id").alias(
            "seller_id_original"
        ),
        F.col("seller_name").alias(
            "seller_name_original"
        ),
        F.col("canal_id").alias(
            "channel_id_original"
        ),
        F.col("regional_code").alias(
            "regional_code_original"
        ),
        F.col("hire_date").alias(
            "hire_date_original"
        ),
        F.col("status").alias(
            "seller_status_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "seller_id",
        normalizar_id(
            F.col("seller_id_original")
        )
    )
    .withColumn(
        "seller_name",
        F.trim(
            F.col("seller_name_original")
        )
    )
    .withColumn(
        "channel_id",
        normalizar_id(
            F.col("channel_id_original")
        )
    )
    .withColumn(
        "regional_code",
        normalizar_codigo_regiao(
            F.col("regional_code_original")
        )
    )
    .withColumn(
        "hire_date",
        converter_timestamp_flexivel_coluna(
            F.col("hire_date_original")
        ).cast("date")
    )
    .withColumn(
        "seller_status",
        normalizar_dominio(
            F.col("seller_status_original")
        )
    )
)

In [0]:
# ============================================================
# RELACIONAR COM CANAIS E REGIOES
# ============================================================

canais_validos = (
    silver_canais
    .select("channel_id")
    .distinct()
    .withColumn(
        "_channel_exists",
        F.lit(True)
    )
)

regioes_validas = (
    silver_regioes
    .select("regional_code")
    .distinct()
    .withColumn(
        "_region_exists",
        F.lit(True)
    )
)

silver_vendedores_base = (
    silver_vendedores_base
    .join(
        canais_validos,
        on="channel_id",
        how="left"
    )
    .join(
        regioes_validas,
        on="regional_code",
        how="left"
    )
)

In [0]:
# ============================================================
# VALIDAR CONFLITOS DE CHAVES
# ============================================================

metricas_vendedores = (
    silver_vendedores_base
    .groupBy("seller_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("seller_name"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_names"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("channel_id"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_channels"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("regional_code"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_regions"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("seller_status"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_statuses"
        )
    )
)

In [0]:
# ============================================================
# CONSOLIDAÇÃO SILVER — VENDEDORES
# ============================================================

silver_vendedores_preparacao = (
    silver_vendedores_base
    .join(
        metricas_vendedores,
        on="seller_id",
        how="left"
    )
    .withColumn(
        "dq_missing_name",
        F.col("seller_name").isNull()
        | (F.trim(F.col("seller_name")) == "")
    )
    .withColumn(
        "dq_suspected_duplicate_name",
        F.upper(
            F.coalesce(
                F.col("seller_name"),
                F.lit("")
            )
        ).contains("DUPLICADO")
    )
    .withColumn(
        "dq_missing_channel",
        F.col("channel_id").isNull()
        | (F.col("channel_id") == "")
    )
    .withColumn(
        "dq_orphan_channel",
        F.col("channel_id").isNotNull()
        & (F.col("channel_id") != "")
        & F.col("_channel_exists").isNull()
    )
    .withColumn(
        "dq_missing_region",
        F.col("regional_code").isNull()
        | (F.col("regional_code") == "")
    )
    .withColumn(
        "dq_orphan_region",
        F.col("regional_code").isNotNull()
        & (F.col("regional_code") != "")
        & F.col("_region_exists").isNull()
    )
    .withColumn(
        "dq_invalid_hire_date",
        F.col("hire_date_original").isNotNull()
        & (
            F.trim(
                F.col("hire_date_original")
            ) != ""
        )
        & F.col("hire_date").isNull()
    )
    .withColumn(
        "dq_missing_status",
        F.col("seller_status").isNull()
        | (F.col("seller_status") == "")
    )
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_names") > 1
        )
        |
        (
            F.col("_distinct_channels") > 1
        )
        |
        (
            F.col("_distinct_regions") > 1
        )
        |
        (
            F.col("_distinct_statuses") > 1
        )
    )
)

In [0]:
# ============================================================
# VENDEDORES - PONTUAÇÃO DE QUALIDADE
# ============================================================

silver_vendedores_preparacao = (
    silver_vendedores_preparacao
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_name"),
            20
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_suspected_duplicate_name"),
            40
        ).otherwise(0)
        +
        F.when(
            F.col("_channel_exists") == True,
            30
        ).otherwise(0)
        +
        F.when(
            F.col("_region_exists") == True,
            20
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_invalid_hire_date")
            & F.col("hire_date").isNotNull(),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_status"),
            5
        ).otherwise(0)
    )
)

In [0]:
# ============================================================
# VENDEDOR - REGISTROS
# ============================================================

janela_prioridade_vendedor = (
    Window
    .partitionBy("seller_id")
    .orderBy(
        F.desc("_quality_score"),
        F.col("hire_date").desc_nulls_last(),
        F.desc("_ingestion_timestamp_utc"),
        F.asc("_source_record_hash")
    )
)

silver_vendedores = (
    silver_vendedores_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_vendedor
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_TIMESTAMP)
    )
    .drop(
        "_channel_exists",
        "_region_exists",
        "_records_by_key",
        "_distinct_names",
        "_distinct_channels",
        "_distinct_regions",
        "_distinct_statuses",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_vendedores
    .orderBy("seller_id")
)

In [0]:
# ============================================================
# VALIDAÇÃO — VENDEDORES
# ============================================================

quantidade_vendedores_silver = (
    silver_vendedores.count()
)

quantidade_vendedores_distintos_esperada = (
    bronze_vendedores
    .select(
        normalizar_id(
            F.col("seller_id")
        ).alias("seller_id")
    )
    .distinct()
    .count()
)

duplicidades_vendedores_silver = (
    silver_vendedores
    .groupBy("seller_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_vendedores = (
    silver_vendedores
    .filter(
        F.col("seller_id").isNull()
        | (F.col("seller_id") == "")
    )
    .count()
)

print(
    f"Bronze: "
    f"{quantidade_vendedores_bronze}"
)

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_vendedores_distintos_esperada}"
)

print(
    f"Silver: "
    f"{quantidade_vendedores_silver}"
)

print(
    f"Registros consolidados: "
    f"{quantidade_vendedores_bronze - quantidade_vendedores_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_vendedores_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_vendedores}"
)

if (
    quantidade_vendedores_silver
    != quantidade_vendedores_distintos_esperada
):
    raise RuntimeError(
        "A quantidade da Silver de vendedores não corresponde as chaves distintas da Bronze"
    )

if duplicidades_vendedores_silver > 0:
    raise RuntimeError(
        "A Silver de vendedores ainda possui duplicidades"
    )

if chaves_nulas_vendedores > 0:
    raise RuntimeError(
        "A Silver de vendedores possui chaves nulas"
    )

In [0]:
# ============================================================
# CONFERÊNCIA DOS VENDEDORES DUPLICADOS
# ============================================================

display(
    silver_vendedores
    .filter(
        F.col("dq_duplicate_key") == True
    )
    .select(
        "seller_id",
        "seller_name",
        "channel_id",
        "regional_code",
        "hire_date",
        "seller_status",
        "dq_orphan_channel",
        "dq_suspected_duplicate_name",
        "dq_conflicting_duplicate"
    )
    .orderBy("seller_id")
)

In [0]:
# ============================================================
# SILVER — VENDEDORES
# ============================================================

TABELA_SILVER_VENDEDORES = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_vendedores`"
)

(
    silver_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_VENDEDORES)
)

quantidade_vendedores_persistidos = (
    spark.table(TABELA_SILVER_VENDEDORES)
    .count()
)

if (
    quantidade_vendedores_persistidos
    != quantidade_vendedores_silver
):
    raise RuntimeError(
        "A persistência da Silver de vendedores apresentou divergência"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_vendedores"
)

print(
    f"Registros persistidos: "
    f"{quantidade_vendedores_persistidos}"
)

In [0]:
# ============================================================
# FUNÇÃO AUXILIAR — VALORES MONETÁRIOS
# ============================================================

# Converte valores numéricos com ponto ou vírgula decimal, valores como N/A, unknown ou strings vazias retornam NULL
def converter_decimal_flexivel_coluna(coluna):
    valor_original = F.trim(
        coluna.cast("string")
    )

    valor_limpo = F.regexp_replace(
        valor_original,
        r"[^0-9,.\-]",
        ""
    )

    valor_padronizado = (
        F.when(
            valor_limpo.isNull()
            | (valor_limpo == ""),
            F.lit(None).cast("string")
        )
        .when(
            F.instr(valor_limpo, ",") > 0,
            F.regexp_replace(
                F.regexp_replace(
                    valor_limpo,
                    r"\.",
                    ""
                ),
                ",",
                "."
            )
        )
        .otherwise(valor_limpo)
    )

    return valor_padronizado.try_cast(
        "decimal(18,2)"
    )


print("Função monetária disponível.")

In [0]:
# ============================================================
# LEITURA DA BRONZE — PRODUTOS
# ============================================================

bronze_produtos = spark.table(
    f"`{CATALOG}`."
    f"`{SCHEMA_BRONZE}`."
    f"`bronze_produtos`"
)

quantidade_produtos_bronze = (
    bronze_produtos.count()
)

print(
    f"Produtos Bronze: "
    f"{quantidade_produtos_bronze} registros"
)

bronze_produtos.printSchema()

In [0]:
# ============================================================
#  SILVER — PRODUTOS
# ============================================================

silver_produtos_base = (
    bronze_produtos
    .select(
        F.col("product.product_id").alias(
            "product_id_original"
        ),
        F.col("product.name").alias(
            "product_name_original"
        ),
        F.col("product.category").alias(
            "category_original"
        ),
        F.col("product.subcategory").alias(
            "subcategory_original"
        ),
        F.col("product.status").alias(
            "product_status_original"
        ),
        F.col("pricing.list_price").alias(
            "list_price_original"
        ),
        F.col("pricing.currency").alias(
            "currency_original"
        ),
        F.col("attributes.family").alias(
            "family_original"
        ),
        F.col("attributes.tags").alias(
            "tags_original"
        ),
        F.col("updated_at").alias(
            "updated_at_original"
        ),
        F.col("_source_record_hash"),
        F.col("_source_file"),
        F.col("_source_path"),
        F.col("_ingestion_batch_id"),
        F.col("_ingestion_timestamp_utc")
    )
    .withColumn(
        "product_id",
        normalizar_id(
            F.col("product_id_original")
        )
    )
    .withColumn(
        "product_name",
        F.trim(
            F.col("product_name_original")
        )
    )
    .withColumn(
        "category",
        normalizar_dominio(
            F.col("category_original")
        )
    )
    .withColumn(
        "subcategory",
        normalizar_dominio(
            F.col("subcategory_original")
        )
    )
    .withColumn(
        "product_status",
        normalizar_dominio(
            F.col("product_status_original")
        )
    )
    .withColumn(
        "list_price",
        converter_decimal_flexivel_coluna(
            F.col("list_price_original")
        )
    )
    .withColumn(
        "currency_code",
        normalizar_id(
            F.col("currency_original")
        )
    )
    .withColumn(
        "product_family",
        normalizar_dominio(
            F.col("family_original")
        )
    )
    .withColumn(
        "tags",
        F.when(
            F.col("tags_original").isNull(),
            F.expr(
                "CAST(array() AS ARRAY<STRING>)"
            )
        )
        .otherwise(
            F.array_sort(
                F.array_distinct(
                    F.transform(
                        F.col("tags_original"),
                        lambda tag: normalizar_dominio(tag)
                    )
                )
            )
        )
    )
    .withColumn(
        "updated_at",
        converter_timestamp_flexivel_coluna(
            F.col("updated_at_original")
        )
    )
)

In [0]:
# ============================================================
#  PRODUTOS - DUPLICIDADE
# ============================================================

metricas_produtos = (
    silver_produtos_base
    .groupBy("product_id")
    .agg(
        F.count("*").alias(
            "_records_by_key"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("product_name"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_names"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("category"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_categories"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("subcategory"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_subcategories"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("product_status"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_statuses"
        ),
        F.countDistinct(
            F.coalesce(
                F.col("list_price").cast("string"),
                F.lit("__NULL__")
            )
        ).alias(
            "_distinct_prices"
        )
    )
)

In [0]:
# ============================================================
# CONSOLIDAÇÃO SILVER — PRODUTOS
# ============================================================

STATUS_PRODUTO_VALIDOS = [
    "ATIVO",
    "INATIVO",
    "DESCONTINUADO"
]

CATEGORIAS_PRODUTO_VALIDAS = [
    "ASSINATURA",
    "HARDWARE",
    "SERVICOS",
    "SOFTWARE"
]

FAMILIAS_PRODUTO_VALIDAS = [
    "CORE",
    "LEGACY",
    "PREMIUM"
]


silver_produtos_preparacao = (
    silver_produtos_base
    .join(
        metricas_produtos,
        on="product_id",
        how="left"
    )
    .withColumn(
        "dq_missing_name",
        F.col("product_name").isNull()
        | (F.trim(F.col("product_name")) == "")
    )
    .withColumn(
        "dq_missing_category",
        F.col("category").isNull()
        | (F.col("category") == "")
    )
    .withColumn(
        "dq_unexpected_category",
        F.col("category").isNotNull()
        & (
            ~F.col("category").isin(
                CATEGORIAS_PRODUTO_VALIDAS
            )
        )
    )
    .withColumn(
        "dq_missing_subcategory",
        F.col("subcategory").isNull()
        | (F.col("subcategory") == "")
    )
    .withColumn(
        "dq_missing_status",
        F.col("product_status").isNull()
        | (F.col("product_status") == "")
    )
    .withColumn(
        "dq_unexpected_status",
        F.col("product_status").isNotNull()
        & (
            ~F.col("product_status").isin(
                STATUS_PRODUTO_VALIDOS
            )
        )
    )
    .withColumn(
        "dq_missing_family",
        F.col("product_family").isNull()
        | (F.col("product_family") == "")
    )
    .withColumn(
        "dq_unexpected_family",
        F.col("product_family").isNotNull()
        & (
            ~F.col("product_family").isin(
                FAMILIAS_PRODUTO_VALIDAS
            )
        )
    )
    .withColumn(
        "dq_invalid_list_price",
        F.col("list_price_original").isNotNull()
        & (
            F.trim(
                F.col("list_price_original")
            ) != ""
        )
        & F.col("list_price").isNull()
    )
    .withColumn(
        "dq_nonpositive_list_price",
        F.col("list_price").isNotNull()
        & (F.col("list_price") <= 0)
    )
    .withColumn(
        "dq_unexpected_currency",
        F.col("currency_code").isNotNull()
        & (F.col("currency_code") != "BRL")
    )
    .withColumn(
        "dq_invalid_updated_at",
        F.col("updated_at_original").isNotNull()
        & (
            F.trim(
                F.col("updated_at_original")
            ) != ""
        )
        & F.col("updated_at").isNull()
    )
    .withColumn(
        "dq_duplicate_key",
        F.col("_records_by_key") > 1
    )
    .withColumn(
        "dq_conflicting_duplicate",
        (
            F.col("_distinct_names") > 1
        )
        |
        (
            F.col("_distinct_categories") > 1
        )
        |
        (
            F.col("_distinct_subcategories") > 1
        )
        |
        (
            F.col("_distinct_statuses") > 1
        )
        |
        (
            F.col("_distinct_prices") > 1
        )
    )
)

In [0]:
# ============================================================
#  PRODUTOS - PONTUAÇÃO
# ============================================================

silver_produtos_preparacao = (
    silver_produtos_preparacao
    .withColumn(
        "_quality_score",
        F.when(
            ~F.col("dq_missing_name"),
            20
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_category")
            & ~F.col("dq_unexpected_category"),
            15
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_subcategory"),
            5
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_status")
            & ~F.col("dq_unexpected_status"),
            10
        ).otherwise(0)
        +
        F.when(
            ~F.col("dq_missing_family")
            & ~F.col("dq_unexpected_family"),
            5
        ).otherwise(0)
        +
        F.when(
            F.col("list_price").isNotNull()
            & ~F.col("dq_nonpositive_list_price"),
            25
        ).otherwise(0)
        +
        F.when(
            F.col("currency_code") == "BRL",
            5
        ).otherwise(0)
        +
        F.when(
            F.col("updated_at").isNotNull(),
            20
        ).otherwise(0)
    )
)

In [0]:
# ============================================================
#  PRODUTOS - DEDUPLICAÇÃO
# ============================================================

janela_prioridade_produto = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.desc("_quality_score"),
        F.col("updated_at").desc_nulls_last(),
        F.desc("_ingestion_timestamp_utc"),
        F.asc("_source_record_hash")
    )
)


silver_produtos = (
    silver_produtos_preparacao
    .withColumn(
        "_deduplication_rank",
        F.row_number().over(
            janela_prioridade_produto
        )
    )
    .filter(
        F.col("_deduplication_rank") == 1
    )
    .withColumn(
        "_silver_processed_at_utc",
        F.lit(SILVER_TIMESTAMP)
    )
    .drop(
        "_records_by_key",
        "_distinct_names",
        "_distinct_categories",
        "_distinct_subcategories",
        "_distinct_statuses",
        "_distinct_prices",
        "_quality_score",
        "_deduplication_rank"
    )
)

display(
    silver_produtos
    .orderBy("product_id")
)

In [0]:
# ============================================================
# VALIDAÇÃO — PRODUTOS
# ============================================================

quantidade_produtos_silver = (
    silver_produtos.count()
)

quantidade_produtos_distintos_esperada = (
    bronze_produtos
    .select(
        normalizar_id(
            F.col("product.product_id")
        ).alias("product_id")
    )
    .distinct()
    .count()
)

duplicidades_produtos_silver = (
    silver_produtos
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

chaves_nulas_produtos = (
    silver_produtos
    .filter(
        F.col("product_id").isNull()
        | (F.col("product_id") == "")
    )
    .count()
)

print(f"Bronze: {quantidade_produtos_bronze}")

print(
    f"Chaves distintas esperadas: "
    f"{quantidade_produtos_distintos_esperada}"
)

print(f"Silver: {quantidade_produtos_silver}")

print(
    f"Registros consolidados: "
    f"{quantidade_produtos_bronze - quantidade_produtos_silver}"
)

print(
    f"Duplicidades na Silver: "
    f"{duplicidades_produtos_silver}"
)

print(
    f"Chaves nulas na Silver: "
    f"{chaves_nulas_produtos}"
)

if (
    quantidade_produtos_silver
    != quantidade_produtos_distintos_esperada
):
    raise RuntimeError(
        "A quantidade da Silver de produtos não corresponde as chaves distintas da Bronze"
    )

if duplicidades_produtos_silver > 0:
    raise RuntimeError(
        "A Silver de produtos ainda possui duplicidades"
    )

if chaves_nulas_produtos > 0:
    raise RuntimeError(
        "A Silver de produtos possui chaves nulas"
    )

In [0]:
# ============================================================
# VALIDAÇÃO — DUPLICIDADES PRODUTOS
# ============================================================

display(
    silver_produtos
    .filter(
        F.col("dq_duplicate_key") == True
    )
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "product_status",
        "list_price",
        "product_family",
        "updated_at",
        "dq_invalid_list_price",
        "dq_conflicting_duplicate"
    )
    .orderBy("product_id")
)

In [0]:
# ============================================================
# SILVER — PRODUTOS
# ============================================================

TABELA_SILVER_PRODUTOS = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_produtos`"
)

(
    silver_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER_PRODUTOS)
)

quantidade_produtos_persistidos = (
    spark.table(TABELA_SILVER_PRODUTOS)
    .count()
)

if (
    quantidade_produtos_persistidos
    != quantidade_produtos_silver
):
    raise RuntimeError(
        "A persistência da Silver de produtos apresentou divergência"
    )

print(
    f"Tabela criada: "
    f"{CATALOG}.{SCHEMA_SILVER}.silver_produtos"
)

print(
    f"Registros persistidos: "
    f"{quantidade_produtos_persistidos}"
)

In [0]:
# ============================================================
# AUDITORIA CONSOLIDADA DAS DIMENSÕES
# ============================================================

import uuid

SILVER_RUN_ID = str(uuid.uuid4())

configuracao_auditoria = [
    (
        "silver_regioes",
        "bronze_regioes",
        "regional_code"
    ),
    (
        "silver_canais",
        "bronze_canais",
        "channel_id"
    ),
    (
        "silver_clientes",
        "bronze_clientes",
        "customer_id"
    ),
    (
        "silver_vendedores",
        "bronze_vendedores",
        "seller_id"
    ),
    (
        "silver_produtos",
        "bronze_produtos",
        "product_id"
    )
]

resultado_auditoria = []

for (
    tabela_silver,
    tabela_bronze,
    chave_silver
) in configuracao_auditoria:

    df_bronze = spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_BRONZE}`."
        f"`{tabela_bronze}`"
    )

    df_silver = spark.table(
        f"`{CATALOG}`."
        f"`{SCHEMA_SILVER}`."
        f"`{tabela_silver}`"
    )

    quantidade_bronze = df_bronze.count()
    quantidade_silver = df_silver.count()

    duplicidades_saida = (
        df_silver
        .groupBy(chave_silver)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    chaves_nulas_saida = (
        df_silver
        .filter(
            F.col(chave_silver).isNull()
            | (F.col(chave_silver) == "")
        )
        .count()
    )

    status = (
        "SUCESSO"
        if (
            duplicidades_saida == 0
            and chaves_nulas_saida == 0
        )
        else "ERRO"
    )

    resultado_auditoria.append(
        (
            SILVER_RUN_ID,
            tabela_silver,
            quantidade_bronze,
            quantidade_silver,
            quantidade_bronze - quantidade_silver,
            duplicidades_saida,
            chaves_nulas_saida,
            status,
            SILVER_TIMESTAMP
        )
    )


schema_auditoria_silver = """
    silver_run_id STRING,
    tabela STRING,
    quantidade_bronze LONG,
    quantidade_silver LONG,
    registros_consolidados LONG,
    duplicidades_saida LONG,
    chaves_nulas_saida LONG,
    status STRING,
    processed_at_utc TIMESTAMP
"""

df_auditoria_silver_dimensoes = (
    spark.createDataFrame(
        resultado_auditoria,
        schema=schema_auditoria_silver
    )
    .orderBy("tabela")
)

display(df_auditoria_silver_dimensoes)

In [0]:
TABELA_AUDITORIA_SILVER = (
    f"`{CATALOG}`."
    f"`{SCHEMA_SILVER}`."
    f"`silver_dimension_audit`"
)

(
    df_auditoria_silver_dimensoes.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABELA_AUDITORIA_SILVER)
)

erros_auditoria = (
    df_auditoria_silver_dimensoes
    .filter(F.col("status") != "SUCESSO")
    .count()
)

if erros_auditoria > 0:
    raise RuntimeError(
        "A auditoria identificou erros nas dimensões Silver"
    )

print(
    "Dimensões Silver concluídas e auditadas com sucesso!"
)

# Conclusão das dimensões Silver

Foram construídas e persistidas cinco entidades dimensionais consolidadas:

- silver_regioes
- silver_canais
- silver_clientes
- silver_vendedores
- silver_produtos

As transformações incluíram:

- normalização das chaves de negócio
- padronização de estados, categorias, status e demais domínios
- conversão de datas e valores monetários
- validação de relacionamentos entre vendedores, canais e regiões
- resolução determinística de registros duplicados
- priorização de registros mais completos, válidos e atualizados
- preservação dos valores originais
- criação de indicadores de qualidade
- manutenção dos metadados de rastreabilidade da camada Bronze

Nenhuma dimensão final possui chaves de negócio duplicadas ou nulas.